# Chapter 1.4: Tokenization, Detokenization & Streaming

## Learning Objectives

By the end of this notebook, you will:

1. **Understand** why tokenization is needed and the trade-offs involved
2. **Implement** the BPE (Byte Pair Encoding) algorithm from scratch
3. **Compare** SentencePiece vs tiktoken tokenizers
4. **Analyze** vocabulary composition of popular model tokenizers
5. **Explain** why detokenization is not simply the reverse of tokenization
6. **Handle** Unicode and partial character issues in streaming
7. **Build** a streaming detokenizer

---

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hideak1/vllm_study/blob/main/notebooks/part1/chapter_1.4_tokenization.ipynb)
[![Download Notebook](https://img.shields.io/badge/Download-Notebook-blue)](https://raw.githubusercontent.com/hideak1/vllm_study/main/notebooks/part1/chapter_1.4_tokenization.ipynb)

**How to do the exercises:**
1. **Google Colab (Recommended)**: Click the "Open in Colab" badge above — you get your own copy with free GPU.
2. **Local Jupyter**: Clone the repo, run `./start.sh`, then open notebooks at `http://localhost:8888`.
3. **Exercises**: Look for cells marked with 🏋️ **Exercise** or **Assignment**. Fill in the `TODO` sections and run the cell to check your work.

> **Tip**: In Colab, the notebook is automatically copied to your Drive — your changes are saved there.

In [ ]:
import re
import collections
from typing import List, Dict, Tuple, Optional
import matplotlib.pyplot as plt
import numpy as np

# We'll implement BPE from scratch first, then compare with real tokenizers
print("Ready to explore tokenization!")

---

## 1. Why Tokenization?

### The Problem

Neural networks work with fixed-size vocabularies, but natural language has:
- Infinite possible words (new words, typos, code, URLs, etc.)
- Multiple languages with different scripts
- Special characters, emojis, mathematical notation

### Tokenization Approaches

| Approach | Vocabulary | Pros | Cons |
|----------|-----------|------|------|
| **Word-level** | ~100K-1M words | Semantically meaningful | Can't handle OOV words, huge vocab |
| **Character-level** | ~256 characters | No OOV, tiny vocab | Very long sequences, no word semantics |
| **Subword (BPE/WP)** | ~32K-128K tokens | Best of both worlds | Less intuitive boundaries |
| **Byte-level** | 256 bytes | Universal, no UNK tokens | Longer sequences for non-ASCII |

Modern LLMs use **subword tokenization**, typically BPE (Byte Pair Encoding) or SentencePiece variants.

---

## 2. BPE Algorithm: From Scratch

### 2.1 The Algorithm

BPE starts with individual characters and iteratively merges the most frequent pair:

1. **Initialize** vocabulary with all individual characters
2. **Count** all adjacent pairs in the corpus
3. **Merge** the most frequent pair into a new token
4. **Repeat** steps 2-3 for `num_merges` iterations

The merge rules learned during training are applied during tokenization.

In [ ]:
## Figure: BPE Merge Process Visualization
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np

fig, axes = plt.subplots(5, 1, figsize=(16, 10), gridspec_kw={'hspace': 0.5})

COLORS = ['#E74C3C', '#3498DB', '#27AE60', '#F39C12', '#8E44AD',
          '#1ABC9C', '#E67E22', '#9B59B6', '#2ECC71', '#D35400',
          '#C0392B', '#2980B9', '#16A085', '#F1C40F', '#7F8C8D']

def draw_tokens(ax, tokens, colors, step_label, merge_info=None):
    ax.set_xlim(0, 20)
    ax.set_ylim(0, 2)
    ax.axis('off')
    
    x = 0.5
    for i, (tok, c) in enumerate(zip(tokens, colors)):
        w = max(len(tok) * 0.35, 0.6)
        ax.add_patch(patches.FancyBboxPatch((x, 0.5), w, 1.0, boxstyle="round,pad=0.08",
                     facecolor=c, alpha=0.8, edgecolor='black', linewidth=1))
        ax.text(x + w/2, 1.0, tok, ha='center', va='center', fontsize=9, fontweight='bold',
                color='white' if tok not in [' '] else 'black')
        x += w + 0.1
    
    ax.text(0, 1.0, step_label, fontsize=10, fontweight='bold', va='center',
            transform=ax.transAxes, ha='right',
            bbox=dict(boxstyle='round,pad=0.2', facecolor='lightyellow', edgecolor='orange'))
    
    if merge_info:
        ax.text(0.75, 0.25, merge_info, fontsize=9, style='italic', color='#2C3E50',
                transform=ax.transAxes, ha='center')

# Step-by-step BPE on "the cat sat"
sentence = "the cat sat"

# Step 0: Character level
chars = list(sentence)
char_colors = [COLORS[hash(c) % len(COLORS)] for c in chars]
draw_tokens(axes[0], chars, char_colors, 'Step 0:\nChars',
            'Initial: each character is its own token')

# Step 1: Merge "t" + "h" -> "th" (most frequent pair)
tokens1 = ['th', 'e', ' ', 'c', 'a', 't', ' ', 's', 'a', 't']
colors1 = ['#E74C3C', '#3498DB', '#BDC3C7', '#27AE60', '#F39C12', '#8E44AD',
           '#BDC3C7', '#1ABC9C', '#F39C12', '#8E44AD']
draw_tokens(axes[1], tokens1, colors1, 'Step 1',
            'Merge: "t" + "h" -> "th"  (freq=1)')

# Step 2: Merge "a" + "t" -> "at"
tokens2 = ['th', 'e', ' ', 'c', 'at', ' ', 's', 'at']
colors2 = ['#E74C3C', '#3498DB', '#BDC3C7', '#27AE60', '#D35400',
           '#BDC3C7', '#1ABC9C', '#D35400']
draw_tokens(axes[2], tokens2, colors2, 'Step 2',
            'Merge: "a" + "t" -> "at"  (freq=2)')

# Step 3: Merge "th" + "e" -> "the"
tokens3 = ['the', ' ', 'c', 'at', ' ', 's', 'at']
colors3 = ['#C0392B', '#BDC3C7', '#27AE60', '#D35400',
           '#BDC3C7', '#1ABC9C', '#D35400']
draw_tokens(axes[3], tokens3, colors3, 'Step 3',
            'Merge: "th" + "e" -> "the"  (freq=1)')

# Step 4: Merge "s" + "at" -> "sat"
tokens4 = ['the', ' ', 'c', 'at', ' ', 'sat']
colors4 = ['#C0392B', '#BDC3C7', '#27AE60', '#D35400', '#BDC3C7', '#2980B9']
draw_tokens(axes[4], tokens4, colors4, 'Step 4',
            'Merge: "s" + "at" -> "sat"  (freq=1)')

plt.suptitle('BPE Merge Process: "the cat sat"', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()
print("BPE iteratively merges the most frequent adjacent pair into a new token.")
print("Common words like \'the\' become single tokens; rare words stay as subwords.")

In [ ]:
class SimpleBPE:
    """
    A simple BPE tokenizer implemented from scratch.
    
    This implementation follows the original BPE paper by Sennrich et al. (2016).
    """
    
    def __init__(self):
        self.merges = {}        # (pair) -> merged_token
        self.merge_order = []   # list of merges in order
        self.vocab = {}         # token -> id
        self.inverse_vocab = {} # id -> token
    
    def _get_word_freqs(self, corpus: List[str]) -> Dict[tuple, int]:
        """
        Split text into words and convert each word to a tuple of characters.
        Add end-of-word marker '</w>'.
        """
        word_freqs = collections.Counter()
        for text in corpus:
            words = text.split()
            for word in words:
                # Split word into characters, add end-of-word marker
                char_tuple = tuple(list(word) + ['</w>'])
                word_freqs[char_tuple] += 1
        return word_freqs
    
    def _get_pair_freqs(self, word_freqs: Dict[tuple, int]) -> Dict[tuple, int]:
        """Count frequency of all adjacent pairs across the vocabulary."""
        pair_freqs = collections.Counter()
        for word, freq in word_freqs.items():
            for i in range(len(word) - 1):
                pair = (word[i], word[i + 1])
                pair_freqs[pair] += freq
        return pair_freqs
    
    def _merge_pair(self, word_freqs: Dict[tuple, int], pair: tuple) -> Dict[tuple, int]:
        """Apply a merge rule to all words in the vocabulary."""
        new_word_freqs = {}
        merged = pair[0] + pair[1]  # concatenate the pair
        
        for word, freq in word_freqs.items():
            new_word = []
            i = 0
            while i < len(word):
                if i < len(word) - 1 and word[i] == pair[0] and word[i + 1] == pair[1]:
                    new_word.append(merged)
                    i += 2
                else:
                    new_word.append(word[i])
                    i += 1
            new_word_freqs[tuple(new_word)] = freq
        
        return new_word_freqs
    
    def train(self, corpus: List[str], num_merges: int = 100, verbose: bool = True):
        """
        Train BPE on a corpus.
        
        Args:
            corpus: List of text strings
            num_merges: Number of merge operations to learn
        """
        word_freqs = self._get_word_freqs(corpus)
        
        # Initialize vocabulary with all characters
        all_chars = set()
        for word in word_freqs:
            for char in word:
                all_chars.add(char)
        
        if verbose:
            print(f"Initial vocabulary size: {len(all_chars)}")
            print(f"Initial characters: {sorted(all_chars)}")
            print(f"\nLearning {num_merges} merges...")
        
        for i in range(num_merges):
            pair_freqs = self._get_pair_freqs(word_freqs)
            if not pair_freqs:
                break
            
            best_pair = max(pair_freqs, key=pair_freqs.get)
            best_freq = pair_freqs[best_pair]
            
            merged_token = best_pair[0] + best_pair[1]
            self.merges[best_pair] = merged_token
            self.merge_order.append(best_pair)
            all_chars.add(merged_token)
            
            word_freqs = self._merge_pair(word_freqs, best_pair)
            
            if verbose and (i < 10 or i % 10 == 0):
                print(f"  Merge {i:>3}: '{best_pair[0]}' + '{best_pair[1]}' -> '{merged_token}' "
                      f"(freq={best_freq})")
        
        # Build vocabulary
        self.vocab = {token: idx for idx, token in enumerate(sorted(all_chars))}
        self.inverse_vocab = {idx: token for token, idx in self.vocab.items()}
        
        if verbose:
            print(f"\nFinal vocabulary size: {len(self.vocab)}")
    
    def tokenize(self, text: str) -> List[str]:
        """Tokenize a string using learned BPE merges."""
        words = text.split()
        all_tokens = []
        
        for word in words:
            # Start with character-level split
            tokens = list(word) + ['</w>']
            
            # Apply merges in order
            for pair in self.merge_order:
                new_tokens = []
                i = 0
                while i < len(tokens):
                    if (i < len(tokens) - 1 and 
                        tokens[i] == pair[0] and tokens[i + 1] == pair[1]):
                        new_tokens.append(pair[0] + pair[1])
                        i += 2
                    else:
                        new_tokens.append(tokens[i])
                        i += 1
                tokens = new_tokens
            
            all_tokens.extend(tokens)
        
        return all_tokens
    
    def encode(self, text: str) -> List[int]:
        """Tokenize and convert to IDs."""
        tokens = self.tokenize(text)
        return [self.vocab.get(t, -1) for t in tokens]
    
    def decode(self, ids: List[int]) -> str:
        """Convert IDs back to text."""
        tokens = [self.inverse_vocab.get(i, '?') for i in ids]
        text = ''.join(tokens).replace('</w>', ' ').strip()
        return text


print("SimpleBPE class defined.")

In [ ]:
# Train BPE on a sample corpus

corpus = [
    "the cat sat on the mat",
    "the cat ate the rat",
    "the rat sat on the cat",
    "the dog sat on the log",
    "the dog ate the bone",
    "the cat and the dog sat together",
    "sitting on the mat is the cat",
    "the cat is sitting on the mat",
    "the mat is on the floor",
    "the floor is under the mat",
]

bpe = SimpleBPE()
bpe.train(corpus, num_merges=30, verbose=True)

In [ ]:
# Test tokenization

test_texts = [
    "the cat sat",
    "the dog ate",
    "sitting together",
    "unknown word",  # contains unseen character combinations
]

print("Tokenization Results:")
print("=" * 60)
for text in test_texts:
    tokens = bpe.tokenize(text)
    ids = bpe.encode(text)
    decoded = bpe.decode(ids)
    print(f"\nInput:    '{text}'")
    print(f"Tokens:   {tokens}")
    print(f"IDs:      {ids}")
    print(f"Decoded:  '{decoded}'")
    print(f"Roundtrip: {'PASS' if decoded == text else 'FAIL'}")

---

## 3. Modern BPE: Byte-Level BPE

### The Evolution

The original BPE works on Unicode characters. **Byte-level BPE** (used by GPT-2/3/4, Llama) works on raw bytes:

1. Convert text to bytes (UTF-8 encoding)
2. Start with 256 base tokens (one per byte value)
3. Apply BPE merges on bytes instead of characters

**Advantages**:
- Never produces `<UNK>` tokens (any text can be encoded as bytes)
- Language-agnostic base vocabulary
- Can handle any Unicode, binary data, etc.

**Disadvantage**:
- Multi-byte Unicode characters (e.g., CJK, emoji) become multiple tokens

In [ ]:
# Demonstrate byte-level representation

texts = [
    "Hello",
    "cafe",
    "Tokyo",
    "\u6771\u4eac",   # Tokyo in Japanese kanji
    "\u4f60\u597d",   # "Hello" in Chinese
    "\U0001f600",     # Grinning face emoji
]

print("UTF-8 Byte Representation:")
print("=" * 70)
for text in texts:
    utf8_bytes = text.encode('utf-8')
    print(f"  '{text}' ({len(text)} chars) -> {list(utf8_bytes)} ({len(utf8_bytes)} bytes)")

print("\n\nKey observations:")
print("  - ASCII chars = 1 byte each")
print("  - CJK chars = 3 bytes each")
print("  - Emoji = 4 bytes")
print("  - This means CJK text needs 3x more base tokens than English!")

---

## 4. SentencePiece vs tiktoken

### Comparison

| Feature | SentencePiece | tiktoken |
|---------|--------------|----------|
| Algorithm | Unigram or BPE | BPE |
| Used by | Llama, T5, ALBERT | GPT-3/4, Claude |
| Language | C++ (with Python bindings) | Rust (with Python bindings) |
| Speed | Fast | Very fast |
| Pre-tokenization | Optional | Regex-based (GPT-2 pattern) |
| Byte fallback | Optional | Built-in |
| Normalization | NFKC, etc. | None |

### Key Difference: Pre-tokenization

tiktoken uses a regex pattern to split text into chunks BEFORE applying BPE:
```python
# GPT-4 pre-tokenization pattern:
pat = r"""'(?i:[sdmt]|ll|ve|re)|[^\r\n\p{L}\p{N}]?+\p{L}+|\p{N}{1,3}| ?[^\s\p{L}\p{N}]++[\r\n]*|\s*[\r\n]|\s+(?!\S)|\s+"""
```

This prevents merges across word boundaries, contractions, etc.

In [ ]:
# Try to use tiktoken (GPT tokenizer)
try:
    import tiktoken
    HAS_TIKTOKEN = True
    print("tiktoken is available!")
except ImportError:
    HAS_TIKTOKEN = False
    print("tiktoken not installed. Install with: pip install tiktoken")
    print("Skipping tiktoken demos...")

# Try SentencePiece
try:
    import sentencepiece as spm
    HAS_SENTENCEPIECE = True
    print("sentencepiece is available!")
except ImportError:
    HAS_SENTENCEPIECE = False
    print("sentencepiece not installed. Install with: pip install sentencepiece")

In [ ]:
# Compare tokenization with tiktoken

if HAS_TIKTOKEN:
    # GPT-4 tokenizer
    enc_gpt4 = tiktoken.encoding_for_model("gpt-4")
    # GPT-3.5 / GPT-2 tokenizer
    enc_gpt2 = tiktoken.get_encoding("gpt2")
    
    test_texts = [
        "Hello, how are you today?",
        "The transformer architecture uses self-attention.",
        "def fibonacci(n): return n if n <= 1 else fibonacci(n-1) + fibonacci(n-2)",
        "https://arxiv.org/abs/2301.00774",
        "\u4f60\u597d\u4e16\u754c",  # Chinese: "Hello World"
        "123456789 * 987654321 = ?",
    ]
    
    print("Tokenization Comparison: GPT-2 vs GPT-4")
    print("=" * 80)
    
    for text in test_texts:
        tokens_gpt2 = enc_gpt2.encode(text)
        tokens_gpt4 = enc_gpt4.encode(text)
        
        print(f"\nText: '{text}'")
        print(f"  GPT-2: {len(tokens_gpt2)} tokens: {[enc_gpt2.decode([t]) for t in tokens_gpt2]}")
        print(f"  GPT-4: {len(tokens_gpt4)} tokens: {[enc_gpt4.decode([t]) for t in tokens_gpt4]}")
else:
    print("Skipping tiktoken comparison (not installed).")
    print("Key differences between GPT-2 and GPT-4 tokenizers:")
    print("  - GPT-4 (cl100k_base) has 100K vocab vs GPT-2's 50K vocab")
    print("  - GPT-4 is more efficient for code, URLs, and non-English text")
    print("  - GPT-4 handles whitespace and numbers differently")

In [ ]:
# Vocabulary analysis

if HAS_TIKTOKEN:
    enc = tiktoken.encoding_for_model("gpt-4")
    
    # Analyze token lengths
    all_tokens = []
    for i in range(enc.n_vocab):
        try:
            token_bytes = enc.decode_single_token_bytes(i)
            token_str = token_bytes.decode('utf-8', errors='replace')
            all_tokens.append(token_str)
        except:
            all_tokens.append(f'<special_{i}>')
    
    # Token length distribution
    token_lengths = [len(t) for t in all_tokens if not t.startswith('<special')]
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    axes[0].hist(token_lengths, bins=range(0, max(token_lengths)+2), 
                 color='#3498db', alpha=0.7, edgecolor='white')
    axes[0].set_xlabel('Token Length (characters)', fontsize=12)
    axes[0].set_ylabel('Count', fontsize=12)
    axes[0].set_title(f'GPT-4 Token Length Distribution (vocab={enc.n_vocab:,})', fontsize=14)
    axes[0].set_xlim(0, 25)
    axes[0].grid(True, alpha=0.3)
    
    # Token type analysis
    categories = {'letters_only': 0, 'digits': 0, 'whitespace_prefix': 0, 
                  'punctuation': 0, 'mixed': 0}
    for t in all_tokens:
        if t.startswith('<special'): continue
        stripped = t.strip()
        if stripped.isalpha():
            categories['letters_only'] += 1
        elif stripped.isdigit():
            categories['digits'] += 1
        elif t.startswith(' ') and len(t) > 1:
            categories['whitespace_prefix'] += 1
        elif len(stripped) <= 2 and not stripped.isalnum():
            categories['punctuation'] += 1
        else:
            categories['mixed'] += 1
    
    axes[1].barh(list(categories.keys()), list(categories.values()),
                 color=['#3498db', '#2ecc71', '#e74c3c', '#f39c12', '#9b59b6'], alpha=0.8)
    axes[1].set_xlabel('Count', fontsize=12)
    axes[1].set_title('Token Categories', fontsize=14)
    
    plt.tight_layout()
    plt.show()
    
    print(f"\nVocabulary size: {enc.n_vocab:,}")
    print(f"Average token length: {np.mean(token_lengths):.1f} characters")
    print(f"Median token length: {np.median(token_lengths):.0f} characters")
else:
    print("Install tiktoken for vocabulary analysis.")

---

## 5. The Detokenization Challenge

### 5.1 Why Detokenization Is Not Just Reverse Tokenization

Several issues make detokenization tricky:

1. **Whitespace handling**: Some tokenizers encode leading spaces as part of tokens
2. **Multi-byte characters**: A single Unicode character may span multiple tokens
3. **Special tokens**: `<bos>`, `<eos>`, `<pad>` need to be handled
4. **Streaming**: We need to output text before we have all tokens

### 5.2 The Partial Character Problem

In byte-level BPE, a Chinese character like `\u4f60` ("you" in Chinese) is encoded as 3 bytes: `[228, 189, 160]`. If these bytes end up as separate tokens, we can't decode the character until we have ALL the bytes!

In [ ]:
# Demonstrate the partial character problem

text = "\u4f60\u597d"  # Chinese: "Hello"
utf8_bytes = text.encode('utf-8')

print(f"Text: '{text}'")
print(f"UTF-8 bytes: {list(utf8_bytes)}")
print(f"Byte count: {len(utf8_bytes)}")
print()

# Simulate streaming: what if we receive bytes one at a time?
print("Streaming byte-by-byte:")
buffer = b''
for i, byte in enumerate(utf8_bytes):
    buffer += bytes([byte])
    try:
        decoded = buffer.decode('utf-8')
        print(f"  Byte {i} ({byte:3d}): buffer = {list(buffer)} -> decoded: '{decoded}'")
        buffer = b''  # reset buffer after successful decode
    except UnicodeDecodeError:
        print(f"  Byte {i} ({byte:3d}): buffer = {list(buffer)} -> INCOMPLETE (waiting for more bytes)")

print("\nThis is exactly the problem with streaming token delivery!")
print("We can't output a partial UTF-8 character.")

In [ ]:
# UTF-8 encoding rules

print("UTF-8 Encoding Structure:")
print("=" * 60)
print("  1 byte:  0xxxxxxx                     (U+0000 - U+007F)    ASCII")
print("  2 bytes: 110xxxxx 10xxxxxx             (U+0080 - U+07FF)    Latin, Greek, etc.")
print("  3 bytes: 1110xxxx 10xxxxxx 10xxxxxx    (U+0800 - U+FFFF)    CJK, etc.")
print("  4 bytes: 11110xxx 10xxxxxx 10xxxxxx 10xxxxxx  (U+10000+)    Emoji, etc.")
print()

def utf8_char_length(first_byte: int) -> int:
    """Determine the length of a UTF-8 character from its first byte."""
    if first_byte < 0x80:   return 1
    elif first_byte < 0xC0: return 0  # continuation byte (error if first)
    elif first_byte < 0xE0: return 2
    elif first_byte < 0xF0: return 3
    else:                   return 4

# Examples
examples = [
    ('A', 'ASCII letter'),
    ('\u00e9', 'e with accent (French)'),
    ('\u4e16', 'Chinese character'),
    ('\U0001f600', 'Grinning face emoji'),
]

for char, desc in examples:
    b = char.encode('utf-8')
    length = utf8_char_length(b[0])
    print(f"  '{char}' ({desc}): {list(b)} -> {length} byte(s)")

---

## 6. Building a Streaming Detokenizer

A streaming detokenizer must handle:
1. Partial UTF-8 characters (buffer incomplete bytes)
2. Token-to-text mapping (using the tokenizer's vocabulary)
3. Efficient incremental output

In [ ]:
class StreamingDetokenizer:
    """
    A streaming detokenizer that handles partial UTF-8 characters.
    
    This is similar to what vLLM/SGLang use for streaming responses.
    """
    
    def __init__(self, tokenizer=None):
        """
        Args:
            tokenizer: A tokenizer object with encode/decode methods.
                      If None, uses a simple byte-to-char mapping.
        """
        self.tokenizer = tokenizer
        self.token_buffer = []     # accumulated token IDs
        self.byte_buffer = b''     # incomplete UTF-8 bytes
        self.prev_text = ''        # previously decoded text
    
    def _tokens_to_bytes(self, token_ids: List[int]) -> bytes:
        """Convert token IDs to bytes using the tokenizer."""
        if self.tokenizer is not None:
            text = self.tokenizer.decode(token_ids)
            return text.encode('utf-8')
        else:
            # Fallback: treat token IDs as byte values
            return bytes(token_ids)
    
    def put(self, token_id: int) -> str:
        """
        Add a new token and return any newly decodable text.
        
        Returns:
            New text that can be safely output (complete UTF-8 characters only)
        """
        self.token_buffer.append(token_id)
        
        # Decode all tokens accumulated so far
        if self.tokenizer is not None:
            try:
                full_text = self.tokenizer.decode(self.token_buffer)
            except Exception:
                return ''  # can't decode yet
            
            # Check for incomplete UTF-8 at the end
            full_bytes = full_text.encode('utf-8')
            
            # Find the last complete character
            safe_text = self._get_safe_text(full_bytes)
            
            # Return only the NEW text
            new_text = safe_text[len(self.prev_text):]
            self.prev_text = safe_text
            
            return new_text
        else:
            # Simple byte-level fallback
            return self._put_byte(token_id)
    
    def _put_byte(self, byte_val: int) -> str:
        """Handle a single byte, buffering incomplete UTF-8 sequences."""
        self.byte_buffer += bytes([byte_val])
        
        # Try to decode the buffer
        try:
            text = self.byte_buffer.decode('utf-8')
            self.byte_buffer = b''  # clear buffer on success
            return text
        except UnicodeDecodeError:
            # Check if we might have complete characters at the start
            for i in range(len(self.byte_buffer), 0, -1):
                try:
                    text = self.byte_buffer[:i].decode('utf-8')
                    self.byte_buffer = self.byte_buffer[i:]
                    return text
                except UnicodeDecodeError:
                    continue
            return ''  # buffer contains only incomplete chars
    
    def _get_safe_text(self, data: bytes) -> str:
        """Get the longest prefix that decodes to valid UTF-8."""
        # Try to decode all of it
        try:
            return data.decode('utf-8')
        except UnicodeDecodeError:
            # Find the last valid boundary
            for i in range(len(data), 0, -1):
                try:
                    return data[:i].decode('utf-8')
                except UnicodeDecodeError:
                    continue
            return ''
    
    def finalize(self) -> str:
        """Flush remaining buffer (with replacement for incomplete chars)."""
        if self.byte_buffer:
            text = self.byte_buffer.decode('utf-8', errors='replace')
            self.byte_buffer = b''
            return text
        
        if self.tokenizer and self.token_buffer:
            full_text = self.tokenizer.decode(self.token_buffer)
            remaining = full_text[len(self.prev_text):]
            return remaining
        
        return ''


print("StreamingDetokenizer class defined.")

In [ ]:
# Demo: Streaming with byte-level detokenizer

# Simulate streaming a mixed English/Chinese/emoji string
text = "Hi \u4f60\u597d \U0001f600"
print(f"Full text: '{text}'")
print(f"UTF-8 bytes: {list(text.encode('utf-8'))}")
print()

# Stream byte by byte
detok = StreamingDetokenizer()
utf8_bytes = text.encode('utf-8')

print("Streaming output:")
print("-" * 50)

accumulated = ''
for i, byte in enumerate(utf8_bytes):
    new_text = detok._put_byte(byte)
    accumulated += new_text
    status = f"'{new_text}'" if new_text else '(buffering)'
    print(f"  Byte {i:2d} ({byte:3d}): output = {status:<20} accumulated = '{accumulated}'")

final = detok.finalize()
if final:
    accumulated += final
    print(f"  Final flush: '{final}'")

print(f"\nFinal result: '{accumulated}'")
print(f"Matches original: {accumulated == text}")

In [ ]:
# Demo with tiktoken (if available)

if HAS_TIKTOKEN:
    enc = tiktoken.encoding_for_model("gpt-4")
    
    text = "The transformer model processes tokens efficiently. \u4f60\u597d\uff01"
    token_ids = enc.encode(text)
    
    print(f"Full text: '{text}'")
    print(f"Token IDs: {token_ids}")
    print(f"Token count: {len(token_ids)}")
    print(f"\nTokens:")
    for tid in token_ids:
        token_text = enc.decode([tid])
        print(f"  {tid:>6} -> '{token_text}'")
    
    # Simulate streaming
    print("\nSimulated streaming output:")
    print("-" * 50)
    
    detok = StreamingDetokenizer(tokenizer=enc)
    for i, tid in enumerate(token_ids):
        new_text = detok.put(tid)
        if new_text:
            print(f"  Token {i} (id={tid:>6}): -> '{new_text}'")
        else:
            print(f"  Token {i} (id={tid:>6}): -> (buffering)")
    
    final = detok.finalize()
    if final:
        print(f"  Final flush: -> '{final}'")
else:
    print("Install tiktoken to see this demo.")

---

## 7. Tokenization Efficiency Analysis

Different tokenizers have different efficiencies for different types of text. Fewer tokens for the same text means:
- Faster inference (fewer decode steps)
- Longer effective context window
- Lower cost (API pricing is per-token)

In [ ]:
# Analyze tokenization efficiency

def analyze_efficiency(text: str, tokenizer_name: str = 'gpt-4'):
    """Analyze tokenization efficiency: characters per token."""
    if not HAS_TIKTOKEN:
        return None
    enc = tiktoken.encoding_for_model(tokenizer_name)
    tokens = enc.encode(text)
    
    n_chars = len(text)
    n_tokens = len(tokens)
    ratio = n_chars / n_tokens if n_tokens > 0 else 0
    
    return {
        'n_chars': n_chars,
        'n_tokens': n_tokens,
        'chars_per_token': ratio,
        'tokens': tokens
    }


if HAS_TIKTOKEN:
    test_texts = {
        'English prose': "The quick brown fox jumps over the lazy dog. This is a sample text to test tokenization efficiency across different types of content.",
        'Python code': "def merge_sort(arr):\n    if len(arr) <= 1:\n        return arr\n    mid = len(arr) // 2\n    left = merge_sort(arr[:mid])\n    right = merge_sort(arr[mid:])\n    return merge(left, right)",
        'JSON data': '{"name": "Alice", "age": 30, "scores": [95, 87, 92], "address": {"city": "New York", "zip": "10001"}}',
        'Math notation': 'f(x) = \\sum_{i=1}^{n} w_i * x_i + b, where ||w||_2 <= 1',
        'Chinese text': '\u4eba\u5de5\u667a\u80fd\u662f\u8ba1\u7b97\u673a\u79d1\u5b66\u7684\u4e00\u4e2a\u5206\u652f\uff0c\u5b83\u4f01\u56fe\u4e86\u89e3\u667a\u80fd\u7684\u5b9e\u8d28',
        'URLs': 'https://github.com/vllm-project/vllm/blob/main/vllm/engine/async_llm_engine.py#L123',
        'Numbers': '3.14159265358979323846264338327950288419716939937510',
        'Repetitive': 'the the the the the the the the the the the the the the the',
    }
    
    print(f"{'Text Type':<20} {'Chars':>6} {'Tokens':>7} {'Chars/Token':>12}")
    print("-" * 50)
    
    types = []
    ratios = []
    for text_type, text in test_texts.items():
        result = analyze_efficiency(text)
        print(f"{text_type:<20} {result['n_chars']:>6} {result['n_tokens']:>7} {result['chars_per_token']:>12.2f}")
        types.append(text_type)
        ratios.append(result['chars_per_token'])
    
    # Visualize
    fig, ax = plt.subplots(figsize=(12, 5))
    bars = ax.barh(types, ratios, color=plt.cm.viridis(np.linspace(0.2, 0.8, len(types))))
    ax.set_xlabel('Characters per Token (higher = more efficient)', fontsize=12)
    ax.set_title('Tokenization Efficiency by Text Type (GPT-4)', fontsize=14)
    ax.axvline(x=np.mean(ratios), color='red', linestyle='--', alpha=0.5, label=f'Mean: {np.mean(ratios):.2f}')
    ax.legend()
    plt.tight_layout()
    plt.show()
    
    print("\nEnglish prose is most efficient (common word patterns).")
    print("Chinese text is least efficient (3 bytes per character).")
else:
    print("Install tiktoken for efficiency analysis.")

---

## 8. Special Tokens and Chat Templates

Modern LLMs use special tokens to structure conversations:

```
Llama-2 Chat Template:
<s>[INST] <<SYS>>\nYou are a helpful assistant.\n<</SYS>>\n\nHello! [/INST] Hi there! How can I help you? </s>

ChatML (used by many models):
<|im_start|>system\nYou are a helpful assistant.<|im_end|>\n<|im_start|>user\nHello!<|im_end|>\n<|im_start|>assistant\n
```

These special tokens:
- Are NOT part of the BPE vocabulary (added separately)
- Must be handled specially during tokenization/detokenization
- Should NOT be shown to users in streaming output

In [ ]:
# Chat template examples

def format_llama2_chat(system_prompt, messages):
    """Format messages using Llama-2 chat template."""
    output = f"<s>[INST] <<SYS>>\n{system_prompt}\n<</SYS>>\n\n"
    
    for i, (role, content) in enumerate(messages):
        if role == 'user':
            if i == 0:
                output += f"{content} [/INST] "
            else:
                output += f"<s>[INST] {content} [/INST] "
        elif role == 'assistant':
            output += f"{content} </s>"
    
    return output

def format_chatml(system_prompt, messages):
    """Format messages using ChatML template."""
    output = f"<|im_start|>system\n{system_prompt}<|im_end|>\n"
    
    for role, content in messages:
        output += f"<|im_start|>{role}\n{content}<|im_end|>\n"
    
    output += "<|im_start|>assistant\n"
    return output


system = "You are a helpful AI assistant."
messages = [
    ('user', 'What is the capital of France?'),
    ('assistant', 'The capital of France is Paris.'),
    ('user', 'What about Germany?'),
]

print("Llama-2 Chat Format:")
print("=" * 60)
print(format_llama2_chat(system, messages))
print()
print("ChatML Format:")
print("=" * 60)
print(format_chatml(system, messages))

# Token count comparison
if HAS_TIKTOKEN:
    enc = tiktoken.encoding_for_model('gpt-4')
    llama_tokens = len(enc.encode(format_llama2_chat(system, messages)))
    chatml_tokens = len(enc.encode(format_chatml(system, messages)))
    raw_tokens = len(enc.encode(system + ' ' + ' '.join(c for _, c in messages)))
    
    print(f"\nToken overhead comparison (GPT-4 tokenizer):")
    print(f"  Raw content:  {raw_tokens} tokens")
    print(f"  Llama-2 fmt:  {llama_tokens} tokens (+{llama_tokens - raw_tokens} overhead)")
    print(f"  ChatML fmt:   {chatml_tokens} tokens (+{chatml_tokens - raw_tokens} overhead)")

---

## Exercises

### Exercise 1: Implement BPE Merge from Scratch

Implement a function that performs a single BPE merge step.

In [ ]:
def bpe_merge_step(tokens: List[str]) -> Tuple[List[str], Tuple[str, str], int]:
    """
    Exercise: Perform one BPE merge step.
    
    Given a list of tokens:
    1. Count all adjacent pairs
    2. Find the most frequent pair
    3. Merge all occurrences of that pair
    4. Return the new token list, the merged pair, and its frequency
    
    Example:
        Input:  ['t', 'h', 'e', ' ', 'c', 'a', 't', ' ', 't', 'h', 'e']
        Output: (['th', 'e', ' ', 'c', 'a', 't', ' ', 'th', 'e'], ('t', 'h'), 2)
    """
    # YOUR CODE HERE
    pass

# Test:
# tokens = list('the cat the')
# new_tokens, pair, freq = bpe_merge_step(tokens)
# print(f"Merged pair: {pair} (freq={freq})")
# print(f"Result: {new_tokens}")

### Exercise 2: Token-per-Dollar Calculator

Write a function that, given API pricing and a text corpus, calculates the cost per token for different tokenizers.

In [ ]:
def cost_analysis(text: str, price_per_1k_tokens: float = 0.03):
    """
    Exercise: Calculate the cost of processing text with different tokenizers.
    
    Compare:
    - Character-level (each char = 1 token)
    - Word-level (each word = 1 token)
    - BPE (use our SimpleBPE or tiktoken)
    
    Calculate:
    - Number of tokens for each method
    - Cost at the given price per 1K tokens
    - Characters per dollar
    """
    # YOUR CODE HERE
    pass

# Test:
# cost_analysis("The quick brown fox jumps over the lazy dog.")

### Exercise 3: Robust Streaming Detokenizer

Extend the StreamingDetokenizer to handle:
1. Special token filtering (don't output special tokens)
2. Stop sequences (stop when a specific string is generated)
3. Token-level callback for logging

In [ ]:
class RobustStreamingDetokenizer(StreamingDetokenizer):
    """
    Exercise: Extended streaming detokenizer with:
    - Special token filtering
    - Stop sequence detection
    - Token-level callback
    """
    
    def __init__(self, tokenizer=None, special_tokens=None, stop_sequences=None, 
                 on_token=None):
        super().__init__(tokenizer)
        self.special_tokens = special_tokens or []
        self.stop_sequences = stop_sequences or []
        self.on_token = on_token  # callback(token_id, text_so_far)
        self.stopped = False
        self.full_text = ''
    
    def put(self, token_id: int) -> str:
        """
        Add a token, filter special tokens, check stop sequences.
        
        Return new text, or empty string if:
        - Token is a special token
        - Stop sequence detected (set self.stopped = True)
        - Incomplete UTF-8 character
        """
        # YOUR CODE HERE
        pass

# Test:
# detok = RobustStreamingDetokenizer(
#     special_tokens=['<|im_start|>', '<|im_end|>'],
#     stop_sequences=['\n\n'],
# )

---

## Summary

### Key Takeaways

1. **BPE** builds subword vocabulary by iteratively merging the most frequent character pairs. It balances between word-level and character-level tokenization.

2. **Byte-level BPE** (used by GPT, Llama) operates on raw bytes, guaranteeing no unknown tokens but requiring multi-token encoding for non-ASCII characters.

3. **Tokenization efficiency** varies greatly: English prose gets ~4 chars/token, CJK text gets ~1.5 chars/token, code gets ~2-3 chars/token.

4. **Detokenization challenges** include partial UTF-8 characters during streaming, special token handling, and whitespace management.

5. **Streaming detokenization** requires buffering incomplete characters and only outputting complete, valid UTF-8 text.

6. **Chat templates** add overhead tokens for role markers and special formatting, which must be handled correctly.

### What's Next

In **Chapter 1.5: Latency vs Throughput -- Serving Metrics**, we'll learn how to measure and reason about the performance of LLM serving systems, including TTFT, TPOT, and the fundamental latency-throughput trade-off.